In [3]:
import pandas as pd
import os

# --- 設定: 実際のCSVファイルパスに書き換えてください ---
csv_a = r"C:\Users\kyohe\Aerial_Photo_Classifier\20260124Data\damaged_polygon_point_mapping.csv"  # CSV A: columns -> vector_filename, point_fid
csv_b = r"C:\Users\kyohe\Aerial_Photo_Classifier\20260124Data\Result_XAI\classification_results.csv"  # CSV B: columns -> ファイル名, 分類正誤
out_path = r'C:\Users\kyohe\Aerial_Photo_Classifier\20260124Data\Result\pointwise_recall_analysis.csv'
join_type = "inner"  # 'inner' または 'left' を選択


# 読み込み
try:
    df_a = pd.read_csv(csv_a, encoding='utf-8-sig')
    df_b = pd.read_csv(csv_b, encoding='utf-8-sig')
except FileNotFoundError as e:
    raise FileNotFoundError(f"CSVファイルが見つかりません: {e}")

# 必要な列の存在確認
required_a = {'vector_filename', 'point_fid'}
required_b = {'ファイル名', '分類正誤'}
if not required_a.issubset(set(df_a.columns)):
    raise ValueError(f"CSV A に必要な列がありません。期待: {required_a} 実際: {set(df_a.columns)}")
if not required_b.issubset(set(df_b.columns)):
    raise ValueError(f"CSV B に必要な列がありません。期待: {required_b} 実際: {set(df_b.columns)}")

# 拡張子なしのベース名列を作成
df_a['base_name'] = df_a['vector_filename'].apply(lambda x: os.path.splitext(x)[0])
df_b['base_name'] = df_b['ファイル名'].apply(lambda x: os.path.splitext(x)[0])

# 結合
df_merged = pd.merge(df_a, df_b, left_on='base_name', right_on='base_name', how=join_type)
print(f'結合結果の行数: {len(df_merged)}')

# ポリゴン単位の分類件数確認
print('ポリゴン単位の分類件数:')
print(df_merged['分類正誤'].value_counts(dropna=False))

# ポイント単位で最終判定を決定: '正判定' が1つでもあれば '正解' 、そうでなければ '不正解'
# def _final_judgement(group):
#     if (group['分類正誤'] == '正判定').any():
#         return '正解'
#     else:
#         return '不正解'
# result = df_merged.groupby('point_fid').apply(_final_judgement)

# ポイント単位で最終判定を決定し、新しい列 final_judge として追加
df_merged['final_judge'] = df_merged.groupby('point_fid')['分類正誤'] \
    .transform(lambda x: '正解' if (x == '正判定').any() else '不正解')
df_merged = df_merged.drop_duplicates(subset=['point_fid'])

# 統計表示
print('ポイント単位の最終判定件数:')
# print(result['最終正誤'].value_counts(dropna=False))

# 出力
df_merged.to_csv(out_path, index=False, encoding='utf-8-sig')
print(f'結果を保存しました -> {out_path}')


結合結果の行数: 155
ポリゴン単位の分類件数:
分類正誤
正判定    108
誤判定     47
Name: count, dtype: int64
ポイント単位の最終判定件数:
結果を保存しました -> C:\Users\kyohe\Aerial_Photo_Classifier\20260124Data\Result\pointwise_recall_analysis.csv
